# CRIVO — Transformer próprio (~17M) e leitor

Pesos aleatórios e vocabulário próprio: **nenhum modelo pré-treinado é baixado**.
Use **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4 ou melhor)** e execute as células na ordem.

**O que acontece**
1. **Dados:** a Wikipédia em português inteira (6 partes, SHA-256 conferido), limpa e sem duplicatas, mais diálogos humanos em português (OpenAssistant) e os fatos do acervo do CRIVO. Algo perto de 900 milhões de tokens.
2. **Pré-treino:** 40.000 passos, lote 64, contexto 256. Na T4, a estimativa é de 2 a 3 horas.
3. **Leitor:** ajuste em cima da base, com os exercícios do acervo (com e sem o nome do assunto) e centenas de milhares de exercícios de leitura tirados da Wikipédia.
4. **Prova:** a leitura da ficha decide se o leitor é aprovado (validação do tutor, com margem).

**Drive: sem encher.** O Drive recebe só:
- `checkpoint.pt`: **um único arquivo**, sobrescrito a cada **10.000 passos** e quando o tempo da sessão acaba (~300 MB);
- `base/`: a base pronta (~75 MB);
- `resultado_leitor.zip`: o leitor treinado (~35 MB).

Os dados e os arquivos intermediários ficam no disco do Colab, não no Drive.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, shutil

drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/CRIVO/transformer-leitor')
DRIVE.mkdir(parents=True, exist_ok=True)
CHECKPOINT = DRIVE / 'checkpoint.pt'      # único arquivo, sobrescrito a cada 10.000 passos
BASE = DRIVE / 'base'
RAMO = 'main'                              # versão do CRIVO usada no treino
ROOT = Path('/content/CRIVO')
FONTES = Path('/content/crivo-fontes')     # disco do Colab (não vai para o Drive)
DADOS = Path('/content/crivo-dados')       # disco do Colab (não vai para o Drive)
PASSOS = 40000
CHECKPOINT_A_CADA = 10000                  # nunca menos que 10.000: o script recusa
MAX_HORAS = 3.5                            # pausa com checkpoint antes de o Colab derrubar a sessão

if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', RAMO,
                    'https://github.com/HROSONE/CRIVO.git', str(ROOT)], check=True)
else:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', RAMO], cwd=ROOT, check=True)
    subprocess.run(['git', 'checkout', '-q', 'FETCH_HEAD'], cwd=ROOT, check=True)
print(subprocess.run(['git', 'log', '-1', '--oneline'], cwd=ROOT, capture_output=True, text=True).stdout)
def rodar(*args):
    subprocess.run([sys.executable, '-u', 'scripts/pretreinar_leitor_16m.py', *args,
                    '--fontes', str(FONTES), '--dados', str(DADOS), '--checkpoint', str(CHECKPOINT),
                    '--base', str(BASE), '--passos', str(PASSOS)], cwd=ROOT, check=True)

In [ ]:
# Preserva o PyTorch com CUDA do Colab; só acrescenta o que falta.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'tokenizers>=0.20,<1', 'pyarrow>=15'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ative a GPU: Ambiente de execução → Alterar o tipo de ambiente → GPU.')
print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Dados (~20 a 40 minutos)
Baixa as 6 partes da Wikipédia (~1,6 GB, conferidas por SHA-256), limpa, tira duplicatas, separa 0,5% para validação, aprende o vocabulário próprio e grava os ids no disco do Colab.
Se a sessão cair, esta célula precisa rodar de novo (o disco do Colab é apagado); o checkpoint no Drive continua lá.

In [ ]:
if not (DADOS / 'treino.bin').exists():
    rodar('--etapa', 'dados')
    rodar('--etapa', 'tokenizador')
    rodar('--etapa', 'tokenizar')
print(open(DADOS / 'manifesto.json').read()[:1500])

## 2. Pré-treino (2 a 3 horas na T4)
Retoma sozinho do `checkpoint.pt` do Drive. Grava o checkpoint **só a cada 10.000 passos** (e quando chega a `MAX_HORAS`), sempre no mesmo arquivo.
Se a sessão cair ou pausar por tempo: rode as células 1, 2 e 3 de novo e depois esta.

In [ ]:
subprocess.run([sys.executable, '-u', 'scripts/pretreinar_leitor_16m.py', '--etapa', 'treinar',
                '--fontes', str(FONTES), '--dados', str(DADOS), '--checkpoint', str(CHECKPOINT),
                '--base', str(BASE), '--passos', str(PASSOS), '--lote', '64',
                '--checkpoint-a-cada', str(CHECKPOINT_A_CADA), '--max-horas', str(MAX_HORAS)],
               cwd=ROOT, check=True)

In [ ]:
# Exporta a base (melhor da validação) para o Drive.
rodar('--etapa', 'exportar')
print(open(BASE / 'relatorio.json').read()[:800])

## 3. Leitor (~30 a 60 minutos)
Ajusta o leitor em cima da base: exercícios do acervo (com e sem o nome do assunto, repetidos 5 vezes) e exercícios de leitura de 60 mil artigos da Wikipédia.
Escolhe o melhor passo pela validação do tutor (perguntas que o leitor nunca vê no treino). Tudo no disco do Colab; só o zip final vai para o Drive.

In [ ]:
LEITOR = Path('/content/leitor')
subprocess.run([sys.executable, '-u', 'scripts/treinar_leitor_transformer.py', '--base', str(BASE),
                '--saida', str(LEITOR), '--dispositivo', 'cuda', '--passos', '15000', '--lote', '64',
                '--lr', '5e-5', '--avaliar-a-cada', '1000',
                '--wikipedia', str(DADOS / 'treino.txt'), '--artigos', '60000'], cwd=ROOT, check=True)

In [ ]:
# Prova: a leitura da ficha com e sem o leitor (validação por assunto, margem de 4 fatos).
subprocess.run([sys.executable, '-u', 'scripts/treinar_leitura_ficha.py', '--com-transformer', str(LEITOR),
                '--saida', '/content/leitura_ficha'], cwd=ROOT, check=True)
import json
print(json.dumps(json.load(open(LEITOR / 'meta.json'))['controle'], ensure_ascii=False, indent=1))

In [ ]:
# Resultado pequeno no Drive: leitor + leitura da ficha + relatório da base.
pacote = Path('/content/resultado_leitor')
if pacote.exists():
    shutil.rmtree(pacote)
shutil.copytree(LEITOR, pacote / 'leitor_transformer')
shutil.copytree('/content/leitura_ficha', pacote / 'leitura_ficha')
shutil.copy(BASE / 'relatorio.json', pacote / 'relatorio_base.json')
shutil.make_archive(str(DRIVE / 'resultado_leitor'), 'zip', pacote)
print('Pronto:', DRIVE / 'resultado_leitor.zip')
print('Tamanho da pasta no Drive (MB):', round(sum(p.stat().st_size for p in DRIVE.rglob('*') if p.is_file()) / 1e6))

## 4. Depois
Mande o `resultado_leitor.zip` (está em `MyDrive/CRIVO/transformer-leitor/`). Ele só é ligado no CRIVO depois de passar nos testes congelados (leitura v2 e perguntas sem o nome do assunto) e na bateria.

Quando terminar tudo, você pode apagar o `checkpoint.pt` do Drive: ele só serve para retomar o pré-treino.